# 01 — Data Exploration

Load a single MIT-BIH Arrhythmia Database record and its expert annotations, and inspect the metadata every later stage depends on. No filtering or ML yet.

## 1-2. Load the record and its annotations

`wfdb.rdrecord` loads the physical signal. `wfdb.rdann` loads a *separate* file of cardiologist-reviewed beat annotations — ground truth we did not compute ourselves.

In [2]:
import wfdb
import collections

RECORD_NAME = "100"

record = wfdb.rdrecord(RECORD_NAME, pn_dir="mitdb")
annotation = wfdb.rdann(RECORD_NAME, "atr", pn_dir="mitdb")

## 3-5. Sampling frequency, signal length, number of channels

- `fs`: samples recorded per second. Converts sample indices to real time everywhere downstream.
- `sig_len`: total number of samples; `sig_len / fs` gives the recording duration in seconds.
- `n_sig`: number of simultaneously recorded ECG leads (electrode placements).

In [3]:
print("Sampling frequency (Hz):", record.fs)
print("Signal length (samples):", record.sig_len)
print("Number of channels:", record.n_sig)
print("Channel names:", record.sig_name)
print("Units:", record.units)
print("Duration (seconds):", record.sig_len / record.fs)
print("Duration (minutes):", record.sig_len / record.fs / 60)

Sampling frequency (Hz): 360
Signal length (samples): 650000
Number of channels: 2
Channel names: ['MLII', 'V5']
Units: ['mV', 'mV']
Duration (seconds): 1805.5555555555557
Duration (minutes): 30.092592592592595


## 6. Extract the ECG signal

`record.p_signal` is a NumPy array of shape `(sig_len, n_sig)` in physical units (mV), not raw ADC integers. We'll work with a single lead — MLII when available, since it's the most commonly used lead for beat detection — and keep the other channel unused for now.

In [4]:
print("p_signal shape:", record.p_signal.shape)

lead_index = record.sig_name.index("MLII") if "MLII" in record.sig_name else 0
ecg_signal = record.p_signal[:, lead_index]

print(f"Using lead '{record.sig_name[lead_index]}' (index {lead_index})")
print("ecg_signal shape:", ecg_signal.shape)
print("First 10 samples (mV):", ecg_signal[:10])

p_signal shape: (650000, 2)
Using lead 'MLII' (index 0)
ecg_signal shape: (650000,)
First 10 samples (mV): [-0.145 -0.145 -0.145 -0.145 -0.145 -0.145 -0.145 -0.145 -0.12  -0.135]


## 7. Inspect annotation symbols and locations

- `annotation.sample`: sample index of each annotated event.
- `annotation.symbol`: a one-character code for what was annotated there (e.g. `N` = normal beat, `V` = PVC, `A` = atrial premature beat). Not every symbol marks a beat — some mark rhythm or signal-quality notes; we'll handle that distinction carefully at the labeling stage. For now we're just looking at what's present in this record.

In [5]:
print("Number of annotations:", len(annotation.sample))
print("First 10 annotation sample indices:", annotation.sample[:10])
print("First 10 annotation symbols:", annotation.symbol[:10])
print("Symbol counts:", collections.Counter(annotation.symbol))

Number of annotations: 2274
First 10 annotation sample indices: [  18   77  370  662  946 1231 1515 1809 2044 2402]
First 10 annotation symbols: ['+', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'A', 'N']
Symbol counts: Counter({'N': 2239, 'A': 33, '+': 1, 'V': 1})
